In [ ]:
!nvidia-smi
!git clone https://github.com/NVIDIA/cutlass.git
!./cutlass/python/CuTeDSL/setup.sh --cu12

In [ ]:
!pip install -U git+https://github.com/NTT123/cute-viz.git

In [ ]:
import math
import torch
import sys
import os

# Adjust this path if your 'cutlass' folder is in a different location
cutlass_path = '/content/cutlass/python/CuTeDSL'
if cutlass_path not in sys.path:
    sys.path.append(cutlass_path)

import cutlass
import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack
from cutlass import utils
import cuda.bindings.driver as cuda



Prepare input tensors A, B and output tensor C

In [ ]:
# init using torch
a_major = "m" 
b_major = "k" 
c_major = "n" 
M, N, K = 1024, 1024 ,512

def create_and_permute_tensor(shape, is_major0, device='cuda', dtype=torch.float16):
    tensor = torch.randn(shape, device=device, dtype=dtype)
    if is_major0:
        return tensor.permute(1, 0)
    return tensor

A = create_and_permute_tensor((M, K), a_major == "m", device='cuda', dtype=torch.float16) #[512, 1024]--[K, M] - COl major
B = create_and_permute_tensor((N, K), b_major == "n", device='cuda', dtype=torch.float16) #[1024, 512]--[N, K] - Row major
C = create_and_permute_tensor((M, N), c_major == "m", device='cuda', dtype=torch.float16) #[1024, 1024]--[M, N] - Row major
print(f"A shape: {A.shape}, B shape: {B.shape}, C shape: {C.shape}")

A = from_dlpack(A, assumed_align=16) # assume 16-byte alignment for better performance when loading data
B = from_dlpack(B, assumed_align=16) # assume 16-byte alignment for better performance when loading data
C = from_dlpack(C, assumed_align=16) # assume 16-byte alignment for better performance when loading data


print(f"A major: {utils.LayoutEnum.from_tensor(A)}, B major: {utils.LayoutEnum.from_tensor(B)}, C major: {utils.LayoutEnum.from_tensor(C)}")



In [ ]:
def show_layout(layout, title="Tensor"):
    try:
        from cute_viz import display_layout

        @cute.jit
        def visualize():
            # Create and render a layout to file
            # layout = cute.make_layout( ((16,16),(256,2)), stride=((512,8192),(1,256)))
            # display_layout(layout)
            display_layout(layout)

        visualize()
    except ImportError:
        pass

In [ ]:
show_layout(A, title="Tensor A")

In [61]:
class GemmV1FP16():
    def __init__(self, cta_tiler=(128,128, 8), num_stages=3, num_threads=256):

        self._cta_tiler = cta_tiler
        self._num_stages = num_stages
        self._num_threads = num_threads
        assert num_threads % 16 == 0 and num_threads > 0, "multiples of 16 required for MMA thread layout"
        self._bM, self._bN, self._bK = self._cta_tiler
        assert self._bM % 16 == 0, "multiple of 16 required for tile dimension M"
        assert self._bN % 16 == 0, "multiple of 16 required for tile dimension N"
        assert self._num_stages >= 3, "num_stages must be greater than or equal to 3"

    @cute.jit
    def __call__(
        self, A:cute.Tensor, 
        B:cute.Tensor, 
        C:cute.Tensor,
        epilogue_op: cutlass.Constexpr = lambda x: x,
        stream: cuda.CUstream = cuda.CUstream(cuda.CUstream_flags.CU_STREAM_DEFAULT),


    ):
        self.a_major_mode = utils.LayoutEnum.from_tensor(A) # col major
        self.b_major_mode = utils.LayoutEnum.from_tensor(B) # row major
        self.c_major_mode = utils.LayoutEnum.from_tensor(C) # row major

        #create shared memory layout for A, B and C
        padding_a = 4 if self.a_major_mode == utils.LayoutEnum.ROW_MAJOR else 0
        padding_b = 4 if self.b_major_mode == utils.LayoutEnum.ROW_MAJOR else 0

        sA_layout = cute.make_layout(
            (self._bM, self._bK, self._num_stages),
            stride=(1, (self._bM + padding_a), self._bK * (self._bM + padding_a)),
        )
        sB_layout = cute.make_layout(
            (self._bN, self._bK, self._num_stages),
            stride=(1, (self._bN + padding_b), self._bK * (self._bN + padding_b)),
        )
        print(f"sA layout: {sA_layout}, sB layout: {sB_layout}")
        # show_layout(sA_layout, title="Shared Memory Layout for A")
        # show_layout(sB_layout, title="Shared Memory Layout for B")


        # create copy layouts for A, B and C from global memory to shared memory
        # global memory -> shared memory copies:
        #   - The majorness of tA/tB follows the majorness of gA/gB
        #   - For k-major, these layouts will copy values one-by-one from
        #       from global memory, without vectorizing
        #   - For m/n-major, it will vectorize to a 128bit copy for faster
        #       data transfer between global and shared memory, as long
        #       as the alignment of the tensor allows it. Otherwise, it
        #       defaults to a non-vectorized copy
        

        # thread layout
        tA = cute.make_layout(
           (self._num_threads // self._bK, self._bK), stride=(self._bK, 1) 
        )

        tB = cute.make_layout(
           (self._num_threads // self._bK, self._bK), stride=(self._bK, 1) 
        )
        vA = cute.make_layout((1, 1))
        vB = cute.make_layout((1, 1))
        atom_async_copy_A = cute.make_copy_atom(
            cute.nvgpu.cpasync.CopyG2SOp(),
            A.element_type,
            num_bits_per_copy=A.element_type.width,
        )

        atom_async_copy_B = cute.make_copy_atom(
            cute.nvgpu.cpasync.CopyG2SOp(),
            B.element_type,
            num_bits_per_copy=B.element_type.width,
        )

        if cutlass.const_expr(self.a_major_mode == utils.LayoutEnum.COL_MAJOR):
            num_vectorized = 4 if (A.layout[0].max_alignment % 16 == 0) else 1
            atom_async_copy_A = cute.make_copy_atom(
                cute.nvgpu.cpasync.CopyG2SOp(),
                A.element_type,
                num_bits_per_copy=A.element_type.width * num_vectorized,
            )
            major_mode_size = self._bM // num_vectorized
            tA = cute.make_layout(
                (major_mode_size, self._num_threads // major_mode_size),
                stride=(1, major_mode_size),
            )
            vA = cute.make_layout((num_vectorized, 1))

        if cutlass.const_expr(self.b_major_mode == utils.LayoutEnum.COL_MAJOR):
            num_vectorized = 4 if (B.layout[0].max_alignment % 16 == 0) else 1
            atom_async_copy_B = cute.make_copy_atom(
                cute.nvgpu.cpasync.CopyG2SOp(),
                A.element_type,
                num_bits_per_copy=B.element_type.width * num_vectorized,
            )
            major_mode_size = self._bN // num_vectorized
            tB = cute.make_layout(
                (major_mode_size, self._num_threads // major_mode_size),
                stride=(1, major_mode_size),
            )
            vB = cute.make_layout((num_vectorized, 1))
        
        print('tA', tA)
        print('tB', tB)
        print('vA', vA)
        print('vB', vB)
        tiled_copy_A = cute.make_tiled_copy_tv(atom_async_copy_A, tA, vA)
        tiled_copy_B = cute.make_tiled_copy_tv(atom_async_copy_B, tB, vB)
        print(tiled_copy_A)
        print(tiled_copy_B)

    @cute.kernel    
    def kernel(self, A:cute.Tensor, B:cute.Tensor, C:cute.Tensor):
        pass 

    def _make_layout_ld_AB(self, M, N, K):
        pass

    def _make_layout_st_C(self, M, N, K):
        pass



        

In [62]:
gemm = GemmV1FP16()

gemm(A, B, C)



sA layout: (128,8,3):(1,128,1024), sB layout: (128,8,3):(1,132,1056)
tA (32,8):(1,32)
tB (32,8):(8,1)
vA (4,1):(1,0)
vB (1,1):(0,0)
Tiled Copy
  Tiler MN:        (128:1,8:1)
  TV Layout tiled: (256,4):(4,1)
Copy Atom
  ThrID:           1:0
  TV Layout Src:   (1,4):(0,1)
  TV Layout Dst:   (1,4):(0,1)
  Value type:      f16
Tiled Copy
  Tiler MN:        (32:1,8:1)
  TV Layout tiled: ((8,32),1):((32,1),0)
Copy Atom
  ThrID:           1:0
  TV Layout Src:   (1,1):(0,1)
  TV Layout Dst:   (1,1):(0,1)
  Value type:      f16


In [ ]:
from transformers import Sam3Processor, Sam3Model
import torch
from PIL import Image
import requests

device = "cuda" if torch.cuda.is_available() else "cpu"

model = Sam3Model.from_pretrained("facebook/sam3").to(device)
processor = Sam3Processor.from_pretrained("facebook/sam3")

# Load image
image_url = "http://images.cocodataset.org/val2017/000000077595.jpg"
image = Image.open(requests.get(image_url, stream=True).raw).convert("RGB")

# Segment using text prompt
inputs = processor(images=image, text="ear", return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)

# Post-process results
results = processor.post_process_instance_segmentation(
    outputs,
    threshold=0.5,
    mask_threshold=0.5,
    target_sizes=inputs.get("original_sizes").tolist()
)[0]

print(f"Found {len(results['masks'])} objects")
# Results contain:
# - masks: Binary masks resized to original image size
# - boxes: Bounding boxes in absolute pixel coordinates (xyxy format)
# - scores: Confidence scores
